# Experimento A1: churn sin abstracciones

Este notebook replica **paso a paso** `flows/experimento_flow.py` (y los módulos de `usecase/churn_model/` y `utils/` que usa), pero sin funciones, sin `@task`/`@flow` de Prefect y sin importar nada del proyecto.

Mapa del flujo original:

1. Leer la configuración (`configs/config.yaml`).
2. Cargar el CSV y quedarse con las columnas que se usan.
3. Partir en entrenamiento y prueba (estratificado).
4. Construir el pipeline: preprocesado (escalado + one-hot) + estimador.
5. Entrenar.
6. Evaluar en el conjunto de prueba.
7. Registrar parámetros, métricas y modelo en MLflow.
8. Generar un resumen.

Lo que **no** se replica: el `@flow` y `@task` de Prefect (solo orquestan y registran logs) y `create_markdown_artifact` (el artefacto de Prefect). El resumen final sí se muestra aquí, pero como Markdown en el notebook.

## 0. Librerías

In [ ]:
import os
from pathlib import Path

import mlflow
import pandas as pd
import yaml
from IPython.display import Markdown, display
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Configuración

En el flow, `load_config` lee `configs/config.yaml` y devuelve un diccionario. Aquí hacemos lo mismo directamente con `yaml`.

`RAIZ` es la carpeta `actividad-1/`. Como el notebook está en `notebooks/`, subimos un nivel con `..`. Así las rutas relativas del YAML se resuelven igual que en el flow.

In [ ]:
RAIZ = Path("..").resolve()

with open(RAIZ / "configs" / "config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg

## 2. Cargar datos

El flow hace tres cosas: lee el CSV (con la ruta relativa a la raíz del proyecto), comprueba que existen todas las columnas que necesita y se queda solo con ellas. Las columnas de identificación (`RowNumber`, `CustomerId`, `Surname`) no aparecen en la lista, así que se descartan.

In [ ]:
NUMERICAS = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Complain",
    "Satisfaction Score",
    "Point Earned",
]
CATEGORICAS = ["Geography", "Gender", "Card Type"]
OBJETIVO = "Exited"

FEATURES = NUMERICAS + CATEGORICAS
COLUMNAS = FEATURES + [OBJETIVO]

ruta_csv = (RAIZ / cfg["datos"]["ruta"]).resolve()
df_completo = pd.read_csv(ruta_csv)

faltan = [c for c in COLUMNAS if c not in df_completo.columns]
assert not faltan, f"Al CSV le faltan columnas: {faltan}"

df = df_completo[COLUMNAS].copy()
print("Ruta:", ruta_csv)
print("Forma:", df.shape)
df.head()

## 3. Partición en entrenamiento y prueba

`train_test_split` separa los datos. Dos parámetros importan:

- `test_size` y `random_state` vienen del YAML. Con la misma semilla, la partición siempre es la misma.
- `stratify=df[OBJETIVO]` mantiene la misma proporción de clientes que abandonan (`Exited = 1`) en los dos conjuntos. Así, si solo hay un 20% de bajas, el conjunto de prueba también tendrá aproximadamente ese 20%.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=cfg["division"]["test_size"],
    random_state=cfg["division"]["random_state"],
    stratify=df[OBJETIVO],
)

print("Entrenamiento:", train_df.shape)
print("Prueba:       ", test_df.shape)
print("Proporción de bajas -> total:", round(df[OBJETIVO].mean(), 4),
      "| train:", round(train_df[OBJETIVO].mean(), 4),
      "| test:", round(test_df[OBJETIVO].mean(), 4))

## 4. Pipeline: preprocesado + modelo

El flow elige el estimador según `modelo.tipo` del YAML. Aquí hacemos lo mismo con un `if`.

El preprocesado tiene dos ramas:

- **Numéricas**: `StandardScaler` resta la media y divide entre la desviación típica. Así todas las variables quedan en una escala comparable.
- **Categóricas**: `OneHotEncoder` convierte cada categoría en una columna 0/1. `handle_unknown="ignore"` evita un error si en prueba aparece una categoría que no estaba en entrenamiento.

Un `Pipeline` une el preprocesado y el modelo, de modo que el escalado y el one-hot se ajustan solo con los datos de entrenamiento.

In [ ]:
modelo_tipo = cfg["modelo"]["tipo"]
parametros = cfg["modelo"].get("parametros") or {}

if modelo_tipo == "logistic_regression":
    estimador = LogisticRegression(**parametros)
elif modelo_tipo == "gradient_boosting":
    estimador = GradientBoostingClassifier(**parametros)
else:
    raise ValueError(f"modelo {modelo_tipo!r} no soportado")

preprocesado = ColumnTransformer(
    [
        ("num", StandardScaler(), NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
    ]
)
pipeline = Pipeline([("preprocesado", preprocesado), ("modelo", estimador)])
pipeline

## 5. Entrenar

`fit` hace dos cosas en orden: ajusta el preprocesado (calcula medias, desviaciones y categorías) y después entrena el estimador con los datos transformados. Solo usa `train_df`.

In [ ]:
X_train = train_df[FEATURES]
y_train = train_df[OBJETIVO]

pipeline.fit(X_train, y_train)
print("Entrenado:", modelo_tipo, parametros)

## 6. Evaluar en prueba

`predict_proba` devuelve, para cada cliente, la probabilidad de cada clase. Nos quedamos con la columna `[:, 1]`, que es la probabilidad de abandono.

Para las métricas que necesitan una decisión (accuracy, precision, recall, f1) se convierte la probabilidad en clase con un **umbral**. El valor por defecto es 0.5. El AUC no usa umbral: mide la capacidad del modelo para ordenar a los clientes que abandonan por encima de los que no.

- **Recall**: de los clientes que abandonan, qué fracción detecta el modelo.
- **Precision**: de los que el modelo marca como abandono, qué fracción abandona de verdad.

In [ ]:
X_test = test_df[FEATURES]
y_test = test_df[OBJETIVO]
umbral = 0.5

proba = pipeline.predict_proba(X_test)[:, 1]
pred = (proba >= umbral).astype(int)

metricas = {
    "auc": float(roc_auc_score(y_test, proba)),
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
}
metricas

## 7. Registrar en MLflow

Este paso hace lo que hacían `configurar_mlflow`, `registrar_run` y `log_run`:

1. **Dónde guardar**: si existe la variable de entorno `MLFLOW_TRACKING_URI`, tiene prioridad. Si no, se usa `http://localhost:5000`, el servidor de MLflow que levanta `services/`. No se usa la base SQLite del YAML, porque ese archivo queda en la carpeta del proyecto y el contenedor no lo lee.
2. **Experimento**: `churn-mlops`.
3. **Parámetros**: el tipo de modelo, sus hiperparámetros y la semilla.
4. **Firma y ejemplo de entrada**: `infer_signature` guarda el esquema de entrada y salida, y `input_example` guarda 3 filas de ejemplo. Sirven para validar y documentar el modelo al servirlo más adelante.
5. **Run**: todo lo anterior se registra dentro de un `start_run`.

In [ ]:
uri = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")

mlflow.set_tracking_uri(uri)
mlflow.set_experiment(cfg["mlflow"]["experimento"])

params_mlflow = {
    "modelo": modelo_tipo,
    **parametros,
    "semilla": cfg["division"]["random_state"],
}
signature = infer_signature(X_test, proba)

with mlflow.start_run(run_name="experimento-a1") as run:
    mlflow.log_params(params_mlflow)
    mlflow.log_metrics(metricas)
    info = mlflow.sklearn.log_model(
        pipeline,
        name="model",
        signature=signature,
        input_example=X_test.head(3),
    )
    run_id = run.info.run_id

print("Tracking URI:", uri)
print("Run ID:      ", run_id)
print("Model URI:   ", info.model_uri)

## 8. Resumen

En el flow, este bloque crea el artefacto de Prefect y hace un `print`. Aquí mostramos el mismo texto en Markdown y lo imprimimos.

In [ ]:
resumen = (
    "## Experimento A1\n\n"
    f"- Modelo: `{cfg['modelo']['tipo']}`\n"
    f"- Filas: {len(df)}\n"
    f"- AUC: **{metricas['auc']:.4f}**\n"
    f"- Recall: {metricas['recall']:.4f}\n"
    f"- Run de MLflow: `{run_id}`"
)
display(Markdown(resumen))
print(f"Run de MLflow: {run_id} | AUC = {metricas['auc']:.4f}")